# E and B modes against survey property maps (DP2, all tracts)

Alex Broughton · October 2026. Rubin stack and Butler access required.

## Introduction

Compares the Schirmer-filtered aperture E- and B-mode S/N maps, and their noise, with i-band survey property maps
sampled on the same HEALPix cells (NSIDE = 4096). All tracts with a mass map in output/ are used, except those within
15° of the Galactic plane. A trend of the E mode with a survey property points to a systematic. The B mode should show
no trend with any property.

- Mass maps. One file per tract, output/{tract}_shapeHSM_massmap.fits, written by dp2-shear-map-tract.ipynb. A tract is
  dropped if the median |b| of its cells is 15° or less. Only cells with aperture completeness of at least 0.6 are
  used, the same threshold as for peaks.
- Survey property maps. i-band HealSparse consolidated maps, degraded on read to NSIDE = 4096 and sampled at the mass
  map cell centers. Loading follows _tutorials/survey-property-maps.ipynb.
- Binned means. Cells are correlated over the 18′ aperture, so the error bars on the binned means underestimate the
  true uncertainty.

## 1.0 Setup

### 1.1 Imports and configuration

All tunable quantities live in the CONFIG dictionary.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import skyproj
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack
from scipy.stats import binned_statistic
from tqdm.auto import tqdm

from lsst.daf.butler import Butler

%matplotlib inline
plt.rcParams.update({"font.size": 12})


# Repo layout: _data/ holds external inputs, output/ holds generated tables/catalogs, plots/ holds figures.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "_data"
OUTPUT_DIR = ROOT / "output"
PLOTS_DIR = ROOT / "plots"
for _d in (OUTPUT_DIR, PLOTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)


def save_plot(fig, name):
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")

In [ ]:
CONFIG = {
    "REPO": "/sdf/data/rubin/repo/dp2_prep",  # Butler repository root
    "COLLECTION": "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",  # Input collection
    "SKYMAP": "lsst_cells_v2",  # Skymap name for Butler
    "INSTRUMENT": "LSSTCam",  # Instrument dimension
    "MASSMAP_GLOB": "*_shapeHSM_massmap.fits",  # Per-tract E/B maps in output/ from dp2-shear-map-tract.ipynb
    "NSIDE": 4096,  # Mass map NSIDE; survey property maps are degraded to this on read
    "GAL_B_MIN": 15.0,  # Tracts with median |b| at or below this (deg) are dropped
    "COMPLETENESS_MIN": 0.6,  # Minimum aperture completeness, as for peaks
    "BAND": "i",  # Band of the survey property maps (shapes are i-band HSM)
    # Survey property maps: axis label -> dataset type.
    "SP_MAPS": {
        "PSF size (pixel)": "deepCoadd_psf_size_consolidated_map_weighted_mean",
        "PSF $e_1$": "deepCoadd_psf_e1_consolidated_map_weighted_mean",
        "PSF $e_2$": "deepCoadd_psf_e2_consolidated_map_weighted_mean",
        "PSF 5σ mag limit (AB)": "deepCoadd_psf_maglim_consolidated_map_weighted_mean",
        "Exposure time (s)": "deepCoadd_exposure_time_consolidated_map_sum",
        "Sky background (nJy)": "deepCoadd_sky_background_consolidated_map_weighted_mean",
    },
    "N_BINS": 10,  # Quantile bins of each survey property for the binned means
}

In [ ]:
butler = Butler(
    CONFIG["REPO"],
    collections=[CONFIG["COLLECTION"]],
    skymap=CONFIG["SKYMAP"],
    instrument=CONFIG["INSTRUMENT"],
)

## 2.0 Load data

### 2.1 E and B mode maps

One row per evaluated HEALPix cell, with the cell center, aperture values, S/N, and completeness. Each tract's cells are
tagged with the tract number.

In [ ]:
tables, dropped = [], []
for path in tqdm(sorted(OUTPUT_DIR.glob(CONFIG["MASSMAP_GLOB"])), desc="mass maps"):
    tract = int(path.name.split("_")[0])
    t = Table.read(path)
    assert t.meta["NSIDE"] == CONFIG["NSIDE"], path.name
    b = SkyCoord(t["RA"], t["DEC"], unit=u.deg).galactic.b.deg
    if np.median(np.abs(b)) <= CONFIG["GAL_B_MIN"]:
        dropped.append(tract)
        continue
    t = t[(t["COMPLETENESS"] >= CONFIG["COMPLETENESS_MIN"]) & np.isfinite(t["E_SN"])]
    t["TRACT"] = tract
    tables.append(t)

mmap = vstack(tables, metadata_conflicts="silent")
tracts = np.unique(mmap["TRACT"])

ra = np.asarray(mmap["RA"])
dec = np.asarray(mmap["DEC"])
e_sn = np.asarray(mmap["E_SN"])
b_sn = np.asarray(mmap["B_SN"])
noise = np.asarray(mmap["E_ap"]) / e_sn  # Aperture shape-noise error, shared by E and B
ipix = hp.ang2pix(CONFIG["NSIDE"], ra, dec, lonlat=True)  # RING, as in the mass maps

print(f"{len(tracts)} tracts kept, {len(dropped)} dropped within |b| <= {CONFIG['GAL_B_MIN']}°: {dropped}")
print(f"{len(mmap):,} cells with completeness >= {CONFIG['COMPLETENESS_MIN']}")

### 2.2 Survey property maps

Each map is degraded on read to the mass map NSIDE and sampled at the cell centers. Cells outside a map's footprint are
set to NaN.

In [ ]:
sp = {}
for label, dataset_type in CONFIG["SP_MAPS"].items():
    hsp = butler.get(dataset_type, band=CONFIG["BAND"], parameters={"degrade_nside": CONFIG["NSIDE"]})
    valid = hsp.get_values_pos(ra, dec, valid_mask=True)
    sp[label] = np.where(valid, hsp.get_values_pos(ra, dec), np.nan)
    print(f"{dataset_type}: nside_sparse={hsp.nside_sparse}, {valid.mean():.1%} of cells covered")

## 3.0 Survey property maps on the footprint

In [ ]:
def draw_cells(ax, values, label, cmap="viridis"):
    """Draw per-cell values on a McBryde projection; a cell shared by two tracts shows the first value."""
    ok = np.isfinite(values)
    pix, first = np.unique(ipix[ok], return_index=True)
    lo, hi = np.percentile(values[ok], [2, 98])
    proj = skyproj.McBrydeSkyproj(ax=ax)
    proj.draw_hpxpix(CONFIG["NSIDE"], pix, values[ok][first], nest=False, vmin=lo, vmax=hi, cmap=cmap)
    proj.draw_colorbar(label=label)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 13), constrained_layout=True)
for ax, (label, x) in zip(axes.flat, sp.items()):
    draw_cells(ax, x, label)
fig.suptitle(f"{len(tracts)} tracts, {CONFIG['BAND']}-band survey properties, NSIDE={CONFIG['NSIDE']}")
save_plot(fig, "dp2_survey_property_maps")

## 4.0 E and B modes against survey properties

Each panel shows the cell S/N against one survey property, with the mean S/N in quantile bins of the property and the
Pearson correlation coefficient r over all cells.

In [ ]:
def plot_against_sp(y, ylabel):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), constrained_layout=True)
    for ax, (label, x) in zip(axes.flat, sp.items()):
        ok = np.isfinite(x) & np.isfinite(y)
        edges = np.unique(np.quantile(x[ok], np.linspace(0, 1, CONFIG["N_BINS"] + 1)))
        mean, _, _ = binned_statistic(x[ok], y[ok], "mean", edges)
        std, _, _ = binned_statistic(x[ok], y[ok], "std", edges)
        n, _, _ = binned_statistic(x[ok], y[ok], "count", edges)

        ax.hexbin(x[ok], y[ok], gridsize=40, mincnt=1, cmap="Greys", bins="log")
        ax.errorbar(0.5 * (edges[1:] + edges[:-1]), mean, std / np.sqrt(n), fmt="o", color="C3")
        ax.set_xlabel(label)
        ax.set_ylabel(ylabel)
        ax.set_title(f"r = {np.corrcoef(x[ok], y[ok])[0, 1]:.3f}")
    fig.suptitle(f"{len(tracts)} tracts, {ylabel} against {CONFIG['BAND']}-band survey properties")
    return fig

### 4.1 E mode

In [ ]:
fig = plot_against_sp(e_sn, "E-mode S/N")
for ax in fig.axes:
    ax.axhline(0.0, color="k", linewidth=0.8, linestyle="--")
save_plot(fig, "dp2_E-mode_vs_survey_properties")

### 4.2 B mode

A trend here indicates a systematic, since lensing produces no B mode.

In [ ]:
fig = plot_against_sp(b_sn, "B-mode S/N")
for ax in fig.axes:
    ax.axhline(0.0, color="k", linewidth=0.8, linestyle="--")
save_plot(fig, "dp2_B-mode_vs_survey_properties")

## 5.0 Noise against survey properties

The shape-noise error of the aperture statistic in each cell, σ = E_ap / E_SN. E and B share the same σ. It is set by
the number, weights, and ellipticities of the sources in the aperture, so it should follow depth and seeing through the
source density. Cells near tract edges or large masks have incomplete apertures and higher σ even above the
completeness cut.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
draw_cells(ax, noise, "Aperture noise σ", cmap="magma")
fig.suptitle(f"{len(tracts)} tracts, aperture noise")
save_plot(fig, "dp2_noise_map")

In [ ]:
fig = plot_against_sp(noise, "Aperture noise σ")
save_plot(fig, "dp2_noise_vs_survey_properties")